# 📰 News Article Summarizer — ML/NLP Project
**Kartik Shrivastava · Hex Softwares Internship · Project 1**

---

This notebook walks through both summarization techniques implemented in this project:

| Mode | Method | Library | Download |
|---|---|---|---|
| **Extractive** | Word-frequency sentence scoring | stdlib only | None |
| **Abstractive** | Seq2seq text generation | HuggingFace Transformers | ~1.6 GB (first run) |

**Run order:** execute cells top-to-bottom. Abstractive cells are clearly marked and can be skipped if `transformers` is not installed.

## 0 · Setup — Add project root to path

In [ ]:
import sys, os

# Make sure the notebook can import from src/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print('Project root:', PROJECT_ROOT)
print('Python:', sys.version)

## 1 · Imports

In [ ]:
from src.extractive_summarizer  import ExtractiveSummarizer
from src.abstractive_summarizer import AbstractiveSummarizer
from src.utils import clean_text, load_article_from_file, save_summary

print('✅ Imports OK')

# Check whether HuggingFace Transformers is installed
abs_model = AbstractiveSummarizer()
print('Transformers available:', abs_model.is_available())

## 2 · Load Sample Articles

In [ ]:
SAMPLE_DIR = os.path.join(PROJECT_ROOT, 'data', 'sample_articles')

articles = {}
for fname in sorted(os.listdir(SAMPLE_DIR)):
    if fname.endswith('.txt'):
        title = fname.replace('_', ' ').replace('.txt', '').title()
        raw   = load_article_from_file(os.path.join(SAMPLE_DIR, fname))
        articles[title] = clean_text(raw)
        print(f'  {title}: {len(articles[title])} chars')

print(f'\nLoaded {len(articles)} articles.')

## 3 · Preview an Article

In [ ]:
DEMO_TITLE  = 'Ai Breakthrough'
DEMO_ARTICLE = articles[DEMO_TITLE]

print(f'=== {DEMO_TITLE} ===')
print(f'Characters : {len(DEMO_ARTICLE)}')
print(f'Words      : {len(DEMO_ARTICLE.split())}')
print()
print(DEMO_ARTICLE)

---
## 4 · Extractive Summarization

**Algorithm:**
1. Split text into sentences on `.!?` boundaries
2. Build a normalised word-frequency table (stop-words excluded)
3. Score each sentence by summing the normalised frequency of its words
4. Pick the top-N sentences and re-join in original reading order

Works 100% offline — no model download needed.

In [ ]:
ext_model = ExtractiveSummarizer(num_sentences=3)
print(ext_model)
print()

In [ ]:
ext_summary = ext_model.summarize(DEMO_ARTICLE)

print('--- EXTRACTIVE SUMMARY ---')
print(ext_summary)
print()
print(f'Original : {len(DEMO_ARTICLE.split())} words')
print(f'Summary  : {len(ext_summary.split())} words')
reduction = (1 - len(ext_summary.split()) / len(DEMO_ARTICLE.split())) * 100
print(f'Reduction: {reduction:.0f}%')

### 4.1 · Run Extractive on All Sample Articles

In [ ]:
for title, text in articles.items():
    summary = ext_model.summarize(text)
    print(f'=== {title} ===')
    print(summary)
    print(f'({len(text.split())} → {len(summary.split())} words)')
    print()

### 4.2 · Vary Number of Sentences

In [ ]:
for n in [1, 2, 3, 5]:
    model   = ExtractiveSummarizer(num_sentences=n)
    summary = model.summarize(DEMO_ARTICLE)
    print(f'--- {n} sentence(s) ({len(summary.split())} words) ---')
    print(summary)
    print()

---
## 5 · Abstractive Summarization (HuggingFace Transformers)

**Model:** `facebook/bart-large-cnn`  
**Size:** ~1.6 GB (downloaded once and cached)  
**Requires:** `pip install transformers torch sentencepiece`

Unlike extractive summarization (which picks existing sentences), abstractive summarization **generates brand-new text** — rephrasing and condensing the article just like a human editor would.

> ⚠️ The cells below will be **skipped automatically** if `transformers` is not installed.

In [ ]:
abs_model = AbstractiveSummarizer()
print('Abstractive model:', abs_model)
print('Transformers installed:', abs_model.is_available())

if not abs_model.is_available():
    print()
    print('⚠️  transformers is not installed.')
    print('Run:  pip install transformers torch sentencepiece')
    print('Then restart this kernel and re-run from here.')

In [ ]:
# This cell is skipped gracefully if transformers is not installed
if abs_model.is_available():
    print('Loading BART model (first run downloads ~1.6 GB)...')
    abs_summary = abs_model.summarize(DEMO_ARTICLE)

    print()
    print('--- ABSTRACTIVE SUMMARY ---')
    print(abs_summary)
    print()
    print(f'Original : {len(DEMO_ARTICLE.split())} words')
    print(f'Summary  : {len(abs_summary.split())} words')
    reduction = (1 - len(abs_summary.split()) / len(DEMO_ARTICLE.split())) * 100
    print(f'Reduction: {reduction:.0f}%')
else:
    print('Skipped — transformers not installed.')

### 5.1 · Side-by-Side Comparison

In [ ]:
if abs_model.is_available():
    for title, text in articles.items():
        ext = ext_model.summarize(text)
        ab  = abs_model.summarize(text)

        print(f'╔══ {title} ═══')
        print(f'║ ORIGINAL   ({len(text.split())} words)')
        print(f'║ {text[:200]}…')
        print(f'╠── EXTRACTIVE ({len(ext.split())} words)')
        print(f'║ {ext}')
        print(f'╠── ABSTRACTIVE ({len(ab.split())} words)')
        print(f'║ {ab}')
        print(f'╚══\n')
else:
    print('Skipped — transformers not installed.')

---
## 6 · Utility Functions

### 6.1 · `clean_text` — HTML stripping & whitespace normalisation

In [ ]:
messy = "<p>Scientists <b>discovered</b> a new \n\n  species!</p>   Extra   spaces.\t\tEnd."
print('Before:', repr(messy))
print('After :', repr(clean_text(messy)))

### 6.2 · `save_summary` — Save results to `outputs/`

In [ ]:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'outputs')

# Save extractive-only
ext_summary = ext_model.summarize(DEMO_ARTICLE)
path1 = save_summary('Ai Breakthrough', ext_summary, OUTPUT_DIR)
print('Saved (extractive only):', path1)

# Save both extractive + abstractive (if available)
if abs_model.is_available():
    ab_summary = abs_model.summarize(DEMO_ARTICLE)
    path2 = save_summary('Ai Breakthrough', ext_summary, OUTPUT_DIR, abstractive=ab_summary)
    print('Saved (both modes)    :', path2)

# Show the file contents
print()
with open(path1, encoding='utf-8') as f:
    print(f.read())

---
## 7 · Error Handling

In [ ]:
# Empty input
try:
    ext_model.summarize('')
except ValueError as e:
    print(f'Empty input → ValueError: {e}')

# Too short for abstractive
try:
    abs_model.summarize('Too short.')
except ValueError as e:
    print(f'Short input → ValueError: {e}')

# Invalid num_sentences
try:
    ExtractiveSummarizer(num_sentences=0)
except ValueError as e:
    print(f'Bad sentences → ValueError: {e}')

---
## 8 · Try Your Own Article

In [ ]:
YOUR_ARTICLE = """
Paste your own news article here and run this cell.
The text should be at least 80 characters long.
Add multiple sentences so the extractive algorithm has something to rank.
"""

text = clean_text(YOUR_ARTICLE)

if len(text) >= 80:
    print('=== EXTRACTIVE ===')
    print(ext_model.summarize(text))

    if abs_model.is_available():
        print()
        print('=== ABSTRACTIVE ===')
        print(abs_model.summarize(text))
else:
    print('Please replace YOUR_ARTICLE with actual article text (min 80 chars).')

---
## 9 · Summary

| | Extractive | Abstractive |
|---|---|---|
| **Method** | Picks existing sentences | Generates new text |
| **Model** | Word-frequency scoring | BART seq2seq transformer |
| **Speed** | Instant | 5–30 seconds (CPU) |
| **Download** | None | ~1.6 GB (first run) |
| **Quality** | Good for factual accuracy | Better for fluency & compression |
| **Offline** | ✅ Always | ✅ After first download |

**Use extractive** when you need speed, offline operation, or exact quotes from the original.  
**Use abstractive** when you need a concise, human-readable condensation that may rephrase content.

---
*Hex Softwares Internship — ML/NLP Project 01*